<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/main/processing_notebooks/02e_processing_naics_nbhd_scatterplot_demographics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import LineString

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
#!wget https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/src/functions.py
import functions as fx

import importlib
importlib.reload(fx)

<module 'functions' from '/content/functions.py'>

In [24]:
gdf = gpd.read_file("/content/drive/MyDrive/C255_final_project/cleaned/open_close_after_2016.geojson")

In [25]:
gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 147785 entries, 0 to 147784
Data columns (total 19 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   uniqueid                           147785 non-null  object        
 1   business_account_number            147785 non-null  int32         
 2   location_id                        147785 non-null  object        
 3   ownership_name                     147785 non-null  object        
 4   dba_name                           147633 non-null  object        
 5   business_start_date                147785 non-null  datetime64[ms]
 6   business_end_date                  75020 non-null   datetime64[ms]
 7   location_start_date                147785 non-null  datetime64[ms]
 8   location_end_date                  107000 non-null  datetime64[ms]
 9   naics_code                         145831 non-null  object        
 10  naics_code_d

## Neighborhood Level Analysis

In [26]:
# Adding SF neighborhood geometry

nbhd_gdf = gpd.read_file(
    "/content/drive/MyDrive/C255_final_project/raw/sf_neighborhoods.geojson"
)

In [27]:
nbhd_gdf.columns

Index([':id', ':version', ':created_at', ':updated_at', 'nhood', 'geometry'], dtype='object')

In [28]:


gdf = gdf.set_crs(epsg=4326)
nbhd_gdf = nbhd_gdf.to_crs(epsg=4326)



In [29]:
gdf = fx.group_points_by_poly_naics_year(
    gdf,
    nbhd_gdf,
    id_col='nhood'
)

In [30]:
gdf

,nhood,naics_group,year,closed,opened,biz_stock
0,Western Addition,Personal Services,2016,3.0,4.0,168
1,Western Addition,Food & Entertainment,2016,4.0,14.0,168
2,Western Addition,Utilities & Construction,2016,4.0,7.0,168
3,Western Addition,Retail,2016,5.0,24.0,168
4,Western Addition,Service,2016,24.0,76.0,168
...,...,...,...,...,...,...
2646,Bayview Hunters Point,Retail,2025,39.0,27.0,320
2647,Bayview Hunters Point,Service,2025,34.0,19.0,320
2648,Bayview Hunters Point,Utilities & Construction,2025,27.0,38.0,320
2649,Bayview Hunters Point,Education & Health,2025,6.0,1.0,320


In [32]:
gdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2651 entries, 0 to 2650
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   nhood        2651 non-null   object 
 1   naics_group  2651 non-null   object 
 2   year         2651 non-null   int32  
 3   closed       2651 non-null   float64
 4   opened       2651 non-null   float64
 5   biz_stock    2651 non-null   int64  
dtypes: float64(2), int32(1), int64(1), object(2)
memory usage: 114.0+ KB


# Baseline by number of businesses in each neighborhood

In [39]:
# closures during COVID by neighborhood and naics_group
closures = (
    gdf[gdf['year'].between(2020, 2021)]
    .groupby(['nhood', 'naics_group'])['closed']
    .sum()
)

# openings during recovery by neighborhood and naics_group
openings = (
    gdf[gdf['year'].between(2022, 2025)]
    .groupby(['nhood', 'naics_group'])['opened']
    .sum()
)

# biz_stock as baseline
biz_stock = (
    gdf
    .groupby(['nhood', 'naics_group'])['biz_stock']
    .first()
)

rate_table = pd.DataFrame({
    'biz_stock': biz_stock,
    'closures_2020_21': closures,
    'openings_2022_25': openings
}).fillna(0)

rate_table['closure_rate'] = rate_table['closures_2020_21'] / rate_table['biz_stock']
rate_table['recovery_rate'] = rate_table['openings_2022_25'] / rate_table['biz_stock']

rate_table

biz_stock  closures_2020_21  \
nhood                 naics_group                                               
Bayview Hunters Point Education & Health                425              11.0   
                      Food & Entertainment              425              45.0   
                      Manufacturing & Industrial        425             144.0   
                      Personal Services                 425              27.0   
                      Retail                            425              85.0   
...                                                     ...               ...   
Western Addition      Manufacturing & Industrial        168              24.0   
                      Personal Services                 168               8.0   
                      Retail                            168              31.0   
                      Service                           168              94.0   
                      Utilities & Construction          168               6.0   

                                                  openings_2022_25  \
nhood                 naics_group                                    
Bayview Hunters Point Education & Health                      27.0   
                      Food & Entertainment                   173.0   
                      Manufacturing & Industrial             179.0   
                      Personal Services                       53.0   
                      Retail                                 156.0   
...                                                            ...   
Western Addition      Manufacturing & Industrial              20.0   
                      Personal Services                       13.0   
                      Retail                                  33.0   
                      Service                                130.0   
                      Utilities & Construction                10.0   

                                                  closure_rate  recovery_rate  
nhood                 naics_group                                              
Bayview Hunters Point Education & Health              0.025882       0.063529  
                      Food & Entertainment            0.105882       0.407059  
                      Manufacturing & Industrial      0.338824       0.421176  
                      Personal Services               0.063529       0.124706  
                      Retail                          0.200000       0.367059  
...                                                        ...            ...  
Western Addition      Manufacturing & Industrial      0.142857       0.119048  
                      Personal Services               0.047619       0.077381  
                      Retail                          0.184524       0.196429  
                      Service                         0.559524       0.773810  
                      Utilities & Construction        0.035714       0.059524  

[282 rows x 5 columns]

## Calculate Business Resilience Rate

In [40]:
# Filtering to nbhds with at least 50 businesses
rate_table_filtered = rate_table[rate_table['biz_stock'] >= 50]

In [43]:
import ipywidgets as widgets
from IPython.display import display

naics_groups = gdf['naics_group'].unique().tolist()
naics_groups.sort()

dropdown = widgets.Dropdown(
    options=['All'] + naics_groups,
    value='All',
    description='Sector:',
)

def show_resilience(naics_group):
    df = rate_table_filtered.copy()

    if naics_group != 'All':
        df = df[df.index.get_level_values('naics_group') == naics_group]

    df['resilience'] = df['recovery_rate'] - df['closure_rate']

    top_10 = df.sort_values('resilience', ascending=False).head(10)
    bottom_10 = df.sort_values('resilience', ascending=True).head(10)

    print("Most Resilient Neighborhoods")
    display(top_10)
    print("Least Resilient Neighborhoods")
    display(bottom_10)

widgets.interact(show_resilience, naics_group=dropdown)

interactive(children=(Dropdown(description='Sector:', options=('All', 'Education & Health', 'Food & Entertainm…

<function __main__.show_resilience(naics_group)>

# Demographic Data by Neighborhood

In [44]:
import geopandas as gpd
from shapely import wkt

# Read in tract to neighborhood geometry file
neighborhoods = pd.read_csv('/content/drive/MyDrive/C255_final_project/raw/tract_to_neighborhood_geom.csv')

# Convert WKT geometry column to shapely geometry and make GeoDataFrame
neighborhoods["geometry"] = neighborhoods["the_geom"].apply(wkt.loads)
neighborhoods = gpd.GeoDataFrame(neighborhoods, geometry="geometry")

In [45]:
neighborhoods = neighborhoods[[
    "geoid",
    "neighborhoods_analysis_boundaries",
    "geometry"
]].rename(columns={
    "geoid": "tract",
    "neighborhoods_analysis_boundaries": "neighborhood"
})

In [46]:
neighborhoods.head()

,tract,neighborhood,geometry
0,6075980900,Bayview Hunters Point,"MULTIPOLYGON (((-122.37276 37.74551, -122.3727..."
1,6075980600,Bayview Hunters Point,"MULTIPOLYGON (((-122.36519 37.73373, -122.3665..."
2,6075980501,McLaren Park,"MULTIPOLYGON (((-122.40667 37.71921, -122.4069..."
3,6075980401,The Farallones,"MULTIPOLYGON (((-123.0036 37.69325, -123.00409..."
4,6075061200,Bayview Hunters Point,"MULTIPOLYGON (((-122.38528 37.74024, -122.3858..."


In [47]:
!pip install census
from census import Census
import pandas as pd

# Prepare Census Bureau API for data pulling
api_key = 'c9ad2fda3c683e4a14992e89daed0afee55738d2'
c = Census(key=api_key)

In [48]:
race_vars = {
    "B03002_001E": "population",
    "B03002_003E": "white",           # Non-Hispanic White
    "B03002_004E": "black",           # Non-Hispanic Black
    "B03002_005E": "aian",            # Non-Hispanic American Indian/Alaska Native
    "B03002_006E": "asian",           # Non-Hispanic Asian
    "B03002_007E": "nhpi",            # Non-Hispanic Native Hawaiian/Pacific Islander
    "B03002_008E": "other",           # Non-Hispanic Some Other Race
    "B03002_012E": "latina_o",        # Hispanic or Latino
}

income_vars = {
    "B19013_001E": "median_income"
}

all_vars = {**race_vars, **income_vars}

# Pulling ACS 5-year 2023 data
acs = pd.DataFrame(
    c.acs5.get(
        list(all_vars.keys()),
        {'for': 'tract:*', 'in': 'state:06 county:075'},
        year=2023
    )
).rename(columns=all_vars)

acs['tract'] = '06075' + acs['tract']

In [49]:
# Replace Census null codes with NaN
acs['median_income'] = acs['median_income'].replace(-666666666, pd.NA)

In [50]:
acs.head()

,population,white,black,aian,asian,nhpi,other,latina_o,median_income,state,county,tract
0,2004.0,770.0,132.0,52.0,608.0,0.0,47.0,336.0,101974.0,06,075,06075010101
1,1795.0,554.0,322.0,0.0,766.0,16.0,0.0,93.0,108417.0,06,075,06075010102
2,2608.0,1762.0,134.0,0.0,206.0,0.0,0.0,311.0,160221.0,06,075,06075010201
3,1761.0,1179.0,58.0,0.0,284.0,27.0,38.0,55.0,206484.0,06,075,06075010202
4,3791.0,2258.0,0.0,0.0,949.0,0.0,28.0,453.0,158973.0,06,075,06075010300


In [51]:
neighborhoods.head()

,tract,neighborhood,geometry
0,6075980900,Bayview Hunters Point,"MULTIPOLYGON (((-122.37276 37.74551, -122.3727..."
1,6075980600,Bayview Hunters Point,"MULTIPOLYGON (((-122.36519 37.73373, -122.3665..."
2,6075980501,McLaren Park,"MULTIPOLYGON (((-122.40667 37.71921, -122.4069..."
3,6075980401,The Farallones,"MULTIPOLYGON (((-123.0036 37.69325, -123.00409..."
4,6075061200,Bayview Hunters Point,"MULTIPOLYGON (((-122.38528 37.74024, -122.3858..."


In [52]:
acs['tract'] = acs['tract'].astype(str).str.lstrip('0')

neighborhoods['tract'] = neighborhoods['tract'].astype(str)
acs['tract'] = acs['tract'].astype(str)

merged = neighborhoods.merge(acs, on='tract', how='inner')

# Merge with neighborhoods shapefile
merged = neighborhoods.merge(acs, on='tract', how='inner')

# Aggregate by neighborhood
agg_cols = {
    'population': 'sum',
    'white': 'sum',
    'black': 'sum',
    'aian': 'sum',
    'asian': 'sum',
    'nhpi': 'sum',
    'other': 'sum',
    'latina_o': 'sum',
    'median_income': 'mean'
}

final = merged.groupby('neighborhood').agg(agg_cols).reset_index()



In [53]:
# Percentages
# combining Native American into other because all the values are less than 1%
# combining NHPI with Asian because all the values are less than .02%
final['other'] = final['other'] + final['aian']
final = final.drop(columns='aian')
final['asian'] = final['asian'] + final['nhpi']
final = final.drop(columns='nhpi')

final['pct_other']   = final['other']   / final['population']
final['pct_white']   = final['white']   / final['population']
final['pct_black']   = final['black']   / final['population']
final['pct_asian']   = final['asian']   / final['population']
final['pct_latina_o']= final['latina_o']/ final['population']

# Filter out very small nbhds
final_df = final[final['population'] >= 200]

final_df

,neighborhood,population,white,black,asian,other,latina_o,median_income,pct_other,pct_white,pct_black,pct_asian,pct_latina_o
0,Bayview Hunters Point,39816.0,3055.0,9295.0,15905.0,31.0,9733.0,96121.0,0.000779,0.076728,0.233449,0.399463,0.244449
1,Bernal Heights,24725.0,10202.0,754.0,3419.0,116.0,8171.0,163135.0,0.004692,0.412619,0.030495,0.138281,0.330475
2,Castro/Upper Market,22024.0,14597.0,552.0,2998.0,173.0,2178.0,202922.857143,0.007855,0.662777,0.025064,0.136124,0.098892
3,Chinatown,12644.0,1239.0,178.0,10282.0,16.0,534.0,41245.0,0.001265,0.097991,0.014078,0.813192,0.042233
4,Excelsior,37915.0,5776.0,582.0,18710.0,408.0,11607.0,110043.875,0.010761,0.152341,0.015350,0.493472,0.306132
5,Financial District/South Beach,24519.0,9394.0,491.0,11296.0,31.0,2282.0,185568.6,0.001264,0.383131,0.020025,0.460704,0.093071
6,Glen Park,8458.0,4630.0,528.0,1260.0,84.0,953.0,181306.5,0.009931,0.547411,0.062426,0.148971,0.112674
8,Haight Ashbury,17780.0,12062.0,472.0,2254.0,111.0,1287.0,197747.0,0.006243,0.678403,0.026547,0.126772,0.072385
9,Hayes Valley,18240.0,9421.0,1335.0,3634.0,545.0,2293.0,133687.6,0.029879,0.516502,0.073191,0.199232,0.125713
10,Inner Richmond,20261.0,9384.0,284.0,7211.0,130.0,1942.0,169155.0,0.006416,0.463156,0.014017,0.355905,0.095849


In [58]:
df = rate_table_filtered.copy()
df['resilience'] = df['recovery_rate'] - df['closure_rate']

final_indexed = final.set_index('neighborhood')
final_indexed['pct_poc'] = 1 - final_indexed['pct_white']
demo_cols = ['median_income', 'pct_poc']

df = df.join(final_indexed[demo_cols], on='nhood', how='left')
df['median_income'] = pd.to_numeric(df['median_income'], errors='coerce')

df = df.sort_values('resilience', ascending=False)


In [59]:
df.head()

,,biz_stock,closures_2020_21,openings_2022_25,closure_rate,recovery_rate,resilience,median_income,pct_poc
nhood,naics_group,,,,,,,,
Castro/Upper Market,Service,254,160.0,486.0,0.629921,1.913386,1.283465,202922.857143,0.337223
Japantown,Education & Health,54,13.0,61.0,0.240741,1.129630,0.888889,95909.000000,0.601118
Glen Park,Service,58,29.0,77.0,0.500000,1.327586,0.827586,181306.500000,0.452589
Portola,Service,57,14.0,60.0,0.245614,1.052632,0.807018,116969.250000,0.894909
Russian Hill,Service,151,120.0,229.0,0.794702,1.516556,0.721854,161533.285714,0.397002


In [61]:
pct_cols = ['neighborhood', 'population', 'median_income',
            'pct_white', 'pct_black', 'pct_asian', 'pct_latina_o', 'pct_other']
final_slim = final_df[pct_cols].set_index('neighborhood')

df_reset = df.reset_index()
df_reset = df_reset.rename(columns={'nhood': 'neighborhood'})

combined = df_reset.merge(final_slim, on='neighborhood', how='inner')

combined = combined.drop(columns='median_income_y').rename(columns={'median_income_x': 'median_income'})

combined.to_csv('sf_business_demographics_nhood_naics.csv')

print(combined.shape)
combined.head()

(238, 16)


,neighborhood,naics_group,biz_stock,closures_2020_21,openings_2022_25,closure_rate,recovery_rate,resilience,median_income,pct_poc,population,pct_white,pct_black,pct_asian,pct_latina_o,pct_other
0,Castro/Upper Market,Service,254,160.0,486.0,0.629921,1.913386,1.283465,202922.857143,0.337223,22024.0,0.662777,0.025064,0.136124,0.098892,0.007855
1,Japantown,Education & Health,54,13.0,61.0,0.240741,1.129630,0.888889,95909.000000,0.601118,3936.0,0.398882,0.039634,0.329268,0.177846,0.014736
2,Glen Park,Service,58,29.0,77.0,0.500000,1.327586,0.827586,181306.500000,0.452589,8458.0,0.547411,0.062426,0.148971,0.112674,0.009931
3,Portola,Service,57,14.0,60.0,0.245614,1.052632,0.807018,116969.250000,0.894909,15558.0,0.105091,0.025582,0.621609,0.209539,0.005142
4,Russian Hill,Service,151,120.0,229.0,0.794702,1.516556,0.721854,161533.285714,0.397002,16879.0,0.602998,0.011908,0.219977,0.111736,0.006102


In [ ]:
# Export File 2: neighborhood_table with per-year open/close counts

open_close_cols = [col for col in neighborhood_table.columns if col.startswith(('open_', 'close_'))]
neighborhood_table[open_close_cols] = neighborhood_table[open_close_cols].fillna(0).astype(int)

neighborhood_table.to_csv('neighborhood_table_with_years.csv')

neighborhood_table = neighborhood_table.join(combined[['median_income', 'pct_poc']], how='left')
neighborhood_table.to_csv('neighborhood_table_with_years.csv', index=True)


# Export file 3: yearly counts by status (for the line chart)
yearly_counts = (
    gdf[gdf['year'] <= 2025]
    .groupby(['year', 'status'])
    .size()
    .reset_index(name='count')
)
yearly_counts.to_csv('netchange_yearly.csv', index=False)

In [ ]:
yearly_counts.head()

In [ ]:
from google.colab import files
files.download('sf_business_demographics_nhood.csv')
files.download('neighborhood_table_with_years.csv')
files.download('netchange_yearly.csv')